In [1]:
from pathlib import Path
import time

import numpy as np
import pandas as pd


In [2]:
RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

average_distance_by_category = pd.read_csv(RAW_DIR / "average_distance_by_category.csv")
hexagon_age_stats = pd.read_csv(RAW_DIR / "hexagon_age_stats.csv")
hexagon_ratio = pd.read_csv(RAW_DIR / "hexagon_ratio.csv")

# Load all raw TACA category files once so category aggregation uses the same input folder.
taca_files = sorted(RAW_DIR.glob("taca_*.csv"))
taca_tables = {path.stem.replace("taca_", ""): pd.read_csv(path) for path in taca_files}


# Data Preparation


### 1.0 Notebook Setup and Inputs
This notebook prepares the base hexagon-level datasets used by later analysis notebooks. It reads raw facility-distance, TACA category, and census-correction files, then produces processed travel-time and corrected-population tables.


## CSV Category Processing


### 1.1 TACA Category Distance Aggregation
This section scans the raw TACA facility CSV files and computes each origin hexagon's average distance to each available facility category. These category averages provide a consistency check against the prebuilt raw distance table used for the travel-time conversion.


In [ ]:
start_time = time.time()
category_averages = []

for category_name, table in taca_tables.items():
    # Only aggregate category files that contain the origin id and network distance fields.
    if {"ori_id", "distance"}.issubset(table.columns):
        category_avg = table.groupby("ori_id")["distance"].mean()
        category_avg.name = f"avg_dist_{category_name}"
        category_averages.append(category_avg)

    if category_averages:
        taca_average_distance = pd.concat(category_averages, axis=1).reset_index()
        taca_average_distance = taca_average_distance.rename(columns={"ori_id": "id"})
    else:
        taca_average_distance = pd.DataFrame()

print(f"Processed {len(category_averages)} TACA category files in {time.time() - start_time:.2f} seconds.")


Processed 2 TACA category files in 0.02 seconds.


## Travel Time Calculation


### 1.2 Facility Distance to Travel Time
The raw distance table stores category-level distances in meters. This section standardizes category names, computes the average facility distance, and converts distance into walking-time estimates for children, working-age adults, and older adults.


In [4]:
# Map raw average-distance column names to the English facility category names used throughout the project.
column_mapping = {
    "id": "Id",
    "avg_dist_restaurant": "restaurant",
    "avg_dist_shopping": "shopping",
    "avg_dist_healthcare": "healthcare",
    "avg_dist_tourist": "tourist",
    "avg_dist_culture": "culture",
    "avg_dist_sports": "sports",
    "avg_dist_transportation": "transportation",
    "avg_dist_life": "life",
    "avg_dist_government": "government",
}

facility_distance = average_distance_by_category.rename(columns=column_mapping).copy()
categories = [
    "restaurant", "shopping", "healthcare", "tourist", "culture",
    "sports", "transportation", "life", "government",
]
facility_distance = facility_distance[["Id"] + categories]
# Average across facility categories to define the all-facility travel-time scenario.
facility_distance["avg_distance"] = facility_distance[categories].mean(axis=1)

# Walking speeds are in meters per second and define the travel-time assumptions for each population group.
speed_child = 1.11
speed_labor = 1.39
speed_elder = 0.75
all_categories = categories + ["avg_distance"]

for category in all_categories:
    facility_distance[f"child_time_{category}"] = facility_distance[category] / speed_child / 60
    facility_distance[f"labor_time_{category}"] = facility_distance[category] / speed_labor / 60
    facility_distance[f"elder_time_{category}"] = facility_distance[category] / speed_elder / 60

# Save category and average travel times for notebook 02 and figure generation.
facility_distance.to_csv(PROCESSED_DIR / "facility_distance_time.csv", index=False, encoding="utf-8")
facility_distance.head()


,Id,restaurant,shopping,healthcare,tourist,culture,sports,transportation,life,government,...,elder_time_transportation,child_time_life,labor_time_life,elder_time_life,child_time_government,labor_time_government,elder_time_government,child_time_avg_distance,labor_time_avg_distance,elder_time_avg_distance
0,0,287.5847,424.4730,333.3470,1305.7023,1240.8126,909.2733,879.1554,871.3015,923.3664,...,19.536787,13.082605,10.447260,19.362256,13.864360,11.071540,20.519253,11.970331,9.559041,17.716089
1,1,67.1598,812.9397,287.3120,274.6357,256.2201,738.0381,857.2119,585.7146,822.1271,...,19.049153,8.794514,7.022957,13.015880,12.344251,9.857639,18.269491,7.843442,6.263468,11.608294
2,2,8860.6860,11085.8044,7935.3437,6973.7433,10175.1822,8751.5937,9167.9246,8207.9043,7122.3663,...,203.731658,123.241806,98.416119,182.397873,106.942437,85.400076,158.274807,130.598179,104.290632,193.285305
3,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,1492.1057,2017.4742,2332.1226,2641.2775,2785.3200,1084.3464,1416.3899,1873.4500,2196.2030,...,31.475331,28.129880,22.463429,41.632222,32.976021,26.333369,48.804511,29.760910,23.765906,44.046146


## Census Correction


### 1.3 Census Population Correction
The gridded population estimates are scaled to match census-derived correction ratios by age group. The output is a corrected hexagon population table used by travel-time analysis, population forecasting, and GWR modeling.


In [5]:
pop_corrected = hexagon_age_stats.copy()
if "geometry" in pop_corrected.columns:
    pop_corrected = pop_corrected.drop(columns=["geometry"])

# Index correction ratios by hexagon Id so age-group columns align before scaling.
ratio_mapper = hexagon_ratio.set_index("Id")
ratio_child = ratio_mapper["ratio_child"]
ratio_labor = ratio_mapper["ratio_labor"]
ratio_old = ratio_mapper["ratio_old"]

# Identify age-group population columns at all anchor years for census scaling.
child_cols = [col for col in pop_corrected.columns if col.startswith("child_") and col.endswith("_sum")]
labor_cols = [col for col in pop_corrected.columns if col.startswith("labor_") and col.endswith("_sum")]
old_cols = [col for col in pop_corrected.columns if col.startswith("old_") and col.endswith("_sum")]

pop_corrected = pop_corrected.set_index("Id")
# Apply the census correction factors separately by age group across all forecast anchor years.
pop_corrected[child_cols] = pop_corrected[child_cols].mul(ratio_child, axis=0)
pop_corrected[labor_cols] = pop_corrected[labor_cols].mul(ratio_labor, axis=0)
pop_corrected[old_cols] = pop_corrected[old_cols].mul(ratio_old, axis=0)
pop_corrected = pop_corrected.reset_index()

# Save the corrected hexagon population table for notebooks 02, 03, and 04.
pop_corrected.to_csv(PROCESSED_DIR / "pop_corrected.csv", index=False, encoding="utf-8")

population_totals = pop_corrected.filter(like="_sum").sum() / 10000
print(population_totals)
pop_corrected.head()


child_2020_sum     262.165558
child_2025_sum     262.165558
child_2030_sum     262.165558
child_2035_sum     262.165558
labor_2020_sum    1383.070587
labor_2025_sum    1393.790906
labor_2030_sum    1389.297079
labor_2035_sum    1307.491262
old_2020_sum        92.463673
old_2025_sum       139.336716
old_2030_sum       223.384791
old_2035_sum       351.089673
dtype: float64


,Id,child_2020_sum,child_2025_sum,child_2030_sum,child_2035_sum,labor_2020_sum,labor_2025_sum,labor_2030_sum,labor_2035_sum,old_2020_sum,old_2025_sum,old_2030_sum,old_2035_sum
0,0,619.396515,619.396515,619.396515,619.396515,2157.250532,2191.845078,2201.543588,2069.525473,223.081048,334.071130,538.497284,855.012444
1,1,1143.617432,1143.617432,1143.617432,1143.617432,3983.020938,4046.894992,4064.801060,3821.050462,411.883761,616.809413,994.249813,1578.644933
2,2,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,3,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,4,268.565445,268.565445,268.565445,268.565445,991.845406,1007.751049,1012.209910,951.511701,95.865872,143.562287,231.411480,367.429336


## Population Merging


### 1.4 Travel-Time and Population Merge
This section joins average travel times with corrected population counts on hexagon Id. It produces a working table with total population by forecast anchor year for downstream accessibility summaries.


In [6]:
# Keep only average travel-time fields before merging corrected population counts.
facility_distance_avg_time = facility_distance[[
    "Id", "child_time_avg_distance", "labor_time_avg_distance", "elder_time_avg_distance"
]].merge(pop_corrected, on="Id", how="left")

# Reconstruct total population at each anchor year from the corrected age-group sums.
for year in [2020, 2025, 2030, 2035]:
    facility_distance_avg_time[f"pop_{year}"] = facility_distance_avg_time[
        [f"child_{year}_sum", f"labor_{year}_sum", f"old_{year}_sum"]
    ].sum(axis=1)

facility_distance_avg_time.head()


,Id,child_time_avg_distance,labor_time_avg_distance,elder_time_avg_distance,child_2020_sum,child_2025_sum,child_2030_sum,child_2035_sum,labor_2020_sum,labor_2025_sum,labor_2030_sum,labor_2035_sum,old_2020_sum,old_2025_sum,old_2030_sum,old_2035_sum,pop_2020,pop_2025,pop_2030,pop_2035
0,0,11.970331,9.559041,17.716089,619.396515,619.396515,619.396515,619.396515,2157.250532,2191.845078,2201.543588,2069.525473,223.081048,334.071130,538.497284,855.012444,2999.728096,3145.312723,3359.437387,3543.934432
1,1,7.843442,6.263468,11.608294,1143.617432,1143.617432,1143.617432,1143.617432,3983.020938,4046.894992,4064.801060,3821.050462,411.883761,616.809413,994.249813,1578.644933,5538.522131,5807.321837,6202.668305,6543.312826
2,2,130.598179,104.290632,193.285305,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,3,NaN,NaN,NaN,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,4,29.760910,23.765906,44.046146,268.565445,268.565445,268.565445,268.565445,991.845406,1007.751049,1012.209910,951.511701,95.865872,143.562287,231.411480,367.429336,1356.276722,1419.878781,1512.186835,1587.506482


## Saved Outputs


### 1.5 Output Inventory
This final section records the processed files produced by the notebook. Later notebooks read these outputs rather than recomputing travel-time conversion or census correction.


In [7]:
print(f"Saved travel-time table: {PROCESSED_DIR / 'facility_distance_time.csv'}")
print(f"Saved corrected population table: {PROCESSED_DIR / 'pop_corrected.csv'}")


Saved travel-time table: ../data/processed/facility_distance_time.csv
Saved corrected population table: ../data/processed/pop_corrected.csv
